# Discussion

This notebook is where we investigate and experiment with ideas for improving the current modeling pipeline. We explore hypotheses, run experiments, inspect results, and let what we learn guide what to try next.

## Label Propagation Diagnostics

Our current pipeline combines feature imputation, graph-based label propagation, and classification with HistGradientBoostingClassifier. The pipeline runs end-to-end, but its cross-validation cost is substantially worse per user than the competition baseline (roughly $68 per validation user versus $38.5 per baseline test user).

Part of this gap is likely explained by a population mismatch rather than model weakness: our labeled data, and therefore every cross-validation fold, consists entirely of manually reviewed ("flagged") users, since `is_cheating` is only observed when `high_conf_clean` is absent. The competition test set, by contrast, does not carry the `high_conf_clean` column at all, suggesting it spans the general population rather than only flagged users. Our validation is therefore estimating `P(cheat | flagged)` on a deliberately hard subpopulation, while the baseline cost is likely averaged over a much easier general population. We separately confirmed this gap is not caused by label propagation or the resolver: training on the originally labeled data alone, with propagation and resolution both disabled, produces cross-validation costs in the same range as every other configuration tried.

That said, label propagation remains worth validating in its own right. Of the pipeline's components, we place the most trust in the classifier and imputer, both of which rely on well-tested scikit-learn functionality applied in a fairly standard way. Label propagation is comparatively bespoke: it required a custom graph construction, a nonstandard use of scikit-learn's `LabelPropagation` via a precomputed kernel, and connected-component filtering logic we wrote ourselves. It is the component most likely to contain either a subtle bug or genuinely weak signal, independent of the score gap discussed above.

To validate it, we will hide labels from held-out users, run propagation using the remaining labeled users and the social graph, and compare the propagated labels with the known ground truth. This will tell us whether the graph is actually providing useful label information and whether the propagation procedure is reliable enough to build upon, before we invest further in improving or relying on it.

**Results.** Label propagation achieved a balanced accuracy of **0.6289** across the five folds, with highly consistent performance (**std = 0.0055**). Since 0.5 corresponds to chance-level performance for balanced accuracy, this indicates that the social graph provides **meaningful but limited predictive signal** for identifying cheating, supporting further use of graph information while suggesting that label propagation alone is unlikely to be sufficient as a predictor. This result is plausible given the nature of the graph: it represents referral relationships, which can reasonably be expected to introduce some association with cheating behavior, but a referral link does not necessarily imply strong similarity in whether a user is cheating. The observed performance is therefore consistent with the graph carrying useful but relatively weak signal. This suggests that **user features should be incorporated into the labeling process for currently unlabeled nodes**, complementing the information provided by the social graph.

## Enhancing Pseudo-Labeling

Our goal is to improve the quality of pseudo-labeling for the currently unlabeled population. We therefore investigate whether combining graph-based label propagation with feature-based classification can improve upon label propagation alone.

The key question is how to combine these sources of information in a way that exploits their complementary signals while preserving a valid evaluation procedure.

### Gated prediction

Label propagation exploits the social graph but does not use the feature information available to the classifier. We therefore consider a heuristic combination of graph-based and feature-based predictions, with the relative weight assigned according to how likely a user is to belong to the flagged population.

Let `F(x)` denote the probability that a user with features `x` is flagged, `CF(x)` the feature-based cheating prediction, and `LP(x)` the graph-based prediction. We define the gated prediction as

$$
G(x) = F(x)\,CF(x) + (1-F(x))\,LP(x).
$$

The flagger `F(x)` therefore acts as a gate between the two predictors: users more likely to be flagged receive greater weight from the feature-based classifier, while users less likely to be flagged rely more heavily on label propagation.

This formulation is a heuristic rather than a probability-theoretic decomposition. Although it resembles the law of total probability, `CF(x)` and `LP(x)` do not represent the corresponding conditional probabilities required for such an interpretation.

The three components also have different data-access and leakage considerations. `F(x)` predicts the observed flagged status and does not depend on the user's `is_cheating` label. `CF(x)` and `LP(x)`, however, both use `is_cheating` labels and must therefore be evaluated without access to the true labels of the users whose predictions are being assessed.